# Session 2 · Working with Real Data I

**Machine Learning Foundations · Sanketana School of Code**

Last session our data was tiny and perfect — because it was made up. Real data never is. Today we open a real file (600 housing listings) and find that some values are simply *missing*. Before any model can learn from a dataset, someone has to find the holes and decide what to do about them.

By the end of this notebook you will be able to:

- load a CSV and take a first, structured look at it
- detect missing values and say exactly how many and where
- decide — with a reason — whether to **drop** or **fill** them
- explain the rule **garbage in = garbage out**

**How this notebook works:** run every cell top to bottom with your coach. Cells marked ✏️ are yours. Every cell already runs even before you fill the gaps — your edits are additive, never blocking.

## Warm-up · Last session's homework

Before any new code, your coach will go through Session 1's homework with you (about 10 minutes — this review happens at the start of every session from now on).

✏️ Keep one thing ready to discuss: in *"predict ___ from ___"* form, which scenario was hardest to frame, and why?

*Your note:*

- 

## Part 1 · Open the file and look

Every real project starts the same way: load the data, then **look** at it before doing anything clever.

The path below is relative to *this notebook's folder*. If it errors, your Jupyter is probably open at the wrong place — open it from the `session-02` folder.

In [ ]:
import pandas as pd

homes = pd.read_csv("../../../datasets/secondary/housing.csv")
homes.head()

In [ ]:
print("rows, columns:", homes.shape)
print("columns:", list(homes.columns))

In [ ]:
homes.info()

### ✏️ Read `.info()`

Look at the **Non-Null Count** column. Most columns say 600. Two of them don't.

1. Which two columns report fewer than 600 non-null values?
2. What does a count below 600 actually mean for those columns?
3. Notice `neighborhood_type` is text (`object`). We are leaving text columns alone today — they get handled next session.

*Your answers:*

1.
2.

## Part 2 · A first numeric summary

`.describe()` summarises every numeric column at once — count, mean, min, max, and the spread. It is the fastest way to sanity-check a dataset.

In [ ]:
homes.describe().round(1)

### ✏️ Read the summary

1. The `count` row is **not** 600 for every column. Which columns fall short — and does that match what `.info()` told you?
2. `price_lakhs` runs from about 18 to 231 lakhs. Does that range seem sensible for homes? (No wrong answer — the habit is to *look*.)

*Your answers:*

1.
2.

## Part 3 · Find the holes, exactly

"Some values are missing" is a feeling. Turn it into a number.

`.isna()` marks every cell `True` (missing) or `False` (present). Summing gives a count per column.

In [ ]:
missing_per_column = homes.isna().sum()
print(missing_per_column)

In [ ]:
# A count means more as a SHARE of all the rows.
# ✏️ TODO: this line already works — read it, then say the percentages out loud.
missing_percent = homes.isna().sum() / len(homes) * 100
print(missing_percent.round(1))

In [ ]:
# Which rows are actually missing age_years? Look at a few.
homes[homes["age_years"].isna()].head()

## Part 4 · Drop or fill?

Two honest options when a value is missing:

- **Drop** the rows with holes. Simple — but you throw away whole listings, and everything else they could have told you.
- **Fill** the holes with a sensible stand-in (often the column's **median**). You keep every row, but you are *inventing* values, so you must choose the stand-in honestly.

There is no universal right answer. It depends on how much is missing and how much each row is worth. Let's try both.

In [ ]:
# Option A: drop every row that is missing ANY value
homes_dropped = homes.dropna()

print("before:                          ", homes.shape[0], "rows")
print("after dropping rows with holes:  ", homes_dropped.shape[0], "rows")
print("rows lost:                       ", homes.shape[0] - homes_dropped.shape[0])

In [ ]:
# Option B: fill the holes with each column's MEDIAN (the middle value).
# The median is less thrown off by extreme values than the mean.
homes_filled = homes.copy()

# ✏️ TODO: these two lines already work. Make sure you see the pattern:
#          take the median, then fillna with it, then RE-ASSIGN the column.
age_median = homes_filled["age_years"].median()
homes_filled["age_years"] = homes_filled["age_years"].fillna(age_median)

dist_median = homes_filled["distance_to_center_km"].median()
homes_filled["distance_to_center_km"] = homes_filled["distance_to_center_km"].fillna(dist_median)

print("missing values after filling:")
print(homes_filled.isna().sum())

### ✏️ Why median, not zero?

1. We filled with the **median**, not `0`. Why would filling an age with `0` be a bad idea? (Hold that thought — Part 5 shows it.)
2. Why might the median be a safer stand-in than the mean?

*Your answers:*

1.
2.

## Part 5 · Garbage in, garbage out

A model only ever knows the data you feed it. Fill a hole carelessly and you do not get "no information" — you get **wrong** information that the model treats as fact.

Watch what a careless fill does to the numbers.

In [ ]:
careless = homes.copy()
careless["age_years"] = careless["age_years"].fillna(0)   # "0 years old" for every unknown age

# The most direct damage: 24 homes we know NOTHING about are now labelled brand-new.
print("homes labelled '0 years old' before the careless fill:", (homes["age_years"] == 0).sum())
print("homes labelled '0 years old' after the careless fill: ", (careless["age_years"] == 0).sum())
print()
print("real mean age (holes ignored): ", round(homes["age_years"].mean(), 1), "years")
print("mean age after filling with 0: ", round(careless["age_years"].mean(), 1), "years  <- dragged down by invented zeros")

### ✏️ Name what just happened

Filling 24 unknown ages with `0` pretends two dozen homes are brand new. The average age of the whole dataset just dropped — not because any home changed, but because we fed the data a lie. Any model that later learns "age vs price" from this now believes something false. **Garbage in, garbage out.**

1. Would filling `distance_to_center_km` with `0` be better or worse than filling `age_years` with `0`? (Hint: what does "0 km from the center" claim about a home?)
2. **Stretch:** for *this* dataset, would you drop the rows or fill them? Defend your choice in one or two sentences.

*Your answers:*

1.
2.

## What we learned

✏️ Three quick reflections — one line each:

1. The first thing you now do when you open a brand-new dataset:
2. One reason to fill instead of drop (or the reverse):
3. Something still unclear (we'll open with it next session):

---

**The path ahead.** Every project in this course follows **data → model → evaluation → insight.** Today was pure *data*: before a model exists, you load it, look at it, and fix what is broken. Clean data is not the boring part — it is the part that decides whether anything you build later can be trusted.

**Next session:** the rest of data prep — turning the text column `neighborhood_type` into numbers, why scaling matters, and meeting **scikit-learn**, the library that builds models for us.

**Homework:** `homework.ipynb`, 30–45 minutes. It states its own success criterion at the top. Revise with `explainer.md`.